In [372]:
# conda activate chronocell

import os, sys
import numpy as np
import pandas as pd

sys.path.append("/mnt/lareaulab/reliscu/programs/FGP_2024")
sys.path.append("/mnt/lareaulab/reliscu/projects/Chronocell/analyses/simulations/code")
sys.path.append("code")

import Chronocell
from reconstruct_RNA_history import *

from protein_from_RNA import *

In [373]:
# Get traj object from running Chronocell
import pickle
with open("eLNPs_var>1.2_traj_WS.pkl", "rb") as f:
    traj = pickle.load(f)

In [7]:
Y = traj.X
Q = traj.Q[:, 0, :] 
tau = traj.tau # State transition times (global)
t = traj.t
theta = traj.theta
topo = traj.topo

theta_ = theta.copy()
a0 = theta_[:, 0] # Starting RNA abundance 
a = theta_[:, 1:len(topo.flatten())] 
beta = theta_[:, -2] # Splicing rate
alpha = a * beta[:, None] # These values are divided by splicing rate; removing this factor now
gamma = theta_[:, -1] # Degradation rate
state_grid = np.searchsorted(tau, t, side="left") - 1


In [ ]:

# U_max = 15
# S_max = 20

# states, index_for = enumerate_states(U_max, S_max)

# # Prep rate matrices

# A_per_gene = [] 
# for j in range(0, alpha.shape[0]):
#     A_for_this_gene = []
#     for i in range(0, alpha.shape[1]):
#         rxns = define_reactions(alpha[j, i], beta[j], gamma[j])
#         A = create_transition_matrix(rxns, U_max, S_max)
#         A_for_this_gene.append(A)
        
#     A_per_gene.append(A_for_this_gene)

In [ ]:
# # Initialize X_fwd with stationary distribution (steady state at t=0)

# pi_per_gene = []

# for j in range(0, alpha.shape[0]):
#     alpha0 = a0[j] * beta[j]
#     rxns0 = define_reactions(alpha0, beta[j], gamma[j])
#     A0 = create_transition_matrix(rxns0, U_max, S_max)
#     pi = stationary_from_transition_matrix(A0)
#     pi_per_gene.append(pi)

In [ ]:
# # Calc forward state probabilities for each gene

# X_fwd_per_gene = []

# for j in range(0, alpha.shape[0]):
#     X_fwd = forward_distribution(A_per_gene[j], pi_per_gene[j], states, t, tau, state_grid)
#     X_fwd_per_gene.append(X_fwd)

In [18]:
U_max = 300 # np.max(Y[:, :, 0]).astype("int") 
S_max = 300 # np.max(Y[:, : , 1]).astype("int")
states, index_for = enumerate_states(U_max, S_max)

In [ ]:
Y[np.argmax(Y[:, :, 0]), j, 0]

np.float64(23.0)

In [ ]:
# j = 0

# A = []
# for i in range(0, alpha.shape[1]):
#     rxns = define_reactions(alpha[j, i], beta[j], gamma[j])
#     A1 = create_transition_matrix(rxns, U_max, S_max)
#     A.append(A1)

In [ ]:
X_bw_per_gene = []
    
for j in range(0, Y.shape[0]):
    print("Starting gene", j)
    
    # Set max # of RNAs based on observed values
    
    U_max = np.max(Y[:, j, 0]) + 3
    S_max = np.max(Y[:, j, 1]) + 3
    states, index_for = enumerate_states(U_max, S_max)
    
    # Make a generator matrix for each transcription rate
    
    A = []
    for i in range(0, alpha.shape[1]):
        rxns = define_reactions(alpha[j, i], beta[j], gamma[j])
        A1 = create_transition_matrix(rxns, U_max, S_max)
        A.append(A1)
    
    # Calculate forward probability distribution
    
    alpha0 = a0[j] * beta[j]
    rxns0 = define_reactions(alpha0, beta[j], gamma[j])
    A0 = create_transition_matrix(rxns0, U_max, S_max)
    pi = stationary_from_transition_matrix(A0) # Distribution at t=0 is stationary distribution ==> steady state
    X_fwd = forward_distribution(A, pi, states, t, tau, state_grid)
    
    # Calculate backward probability distribution (per cell)
    
    X_bw_per_cell = [] 
    for n in range(0, Q.shape[0]):
        X_bw = backward_distribution(Y, A, X_fwd, Q, n, states, index_for, t, tau, state_grid)
        X_bw_per_cell.append(X_bw)
        
    X_bw_per_gene.append(X_bw_per_cell)


In [ ]:
# Y_observed, Y, theta, rd, true_t, true_l = simulate_RNA(topo, tau, theta[0, :][None, :], n=20000, random_seed=666)